In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from scipy.stats import f_oneway, kruskal, zscore
from collections import Counter

sns.set(style="whitegrid")

# ------------------ CONFIG ------------------
TRAIN_PATH = "../data/proteinas_train.csv"
SAVE_DIR = "../eda/eda_outputs"
OUTPUT_DIR = "../preprocessing/preprocessing_artifacts"
MAX_LEN = 300
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------ LOAD & CLEAN ------------------
df = pd.read_csv(TRAIN_PATH)

column_map = {
    "ID_Proteína": "Protein_ID", "Sequência": "Sequence",
    "Massa_Molecular": "Molecular_Mass", "Ponto_Isoelétrico": "Isoelectric_Point",
    "Hidrofobicidade": "Hydrophobicity", "Carga_Total": "Total_Charge",
    "Proporção_Polar": "Polar_Proportion", "Proporção_Apolar": "Nonpolar_Proportion",
    "Comprimento_Sequência": "Sequence_Length", "Classe": "Class"
}

df.rename(columns=column_map, inplace=True)
df.columns = df.columns.str.strip()
df["Class"] = df["Class"].str.strip().astype("category")

numerical_features = [
    "Molecular_Mass", "Isoelectric_Point", "Hydrophobicity",
    "Total_Charge", "Polar_Proportion", "Nonpolar_Proportion", "Sequence_Length"
]

In [9]:
# ------------------ PHASE 1: EDA ------------------

# --- Class Distribution ---
sns.countplot(data=df, x="Class")
plt.title("Class Distribution")
plt.xticks(rotation=45)
plt.savefig(f"{SAVE_DIR}/class_distribution.png")
plt.clf()

# --- Numeric Distributions ---
for col in numerical_features:
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution: {col}")
    plt.savefig(f"{SAVE_DIR}/{col}_distribution.png")
    plt.clf()

# --- Correlation Heatmaps ---
pearson_corr = df[numerical_features].corr("pearson")
sns.heatmap(pearson_corr, annot=True, cmap="coolwarm")
plt.title("Pearson Correlation")
plt.savefig(f"{SAVE_DIR}/correlation_pearson.png")
plt.clf()

spearman_corr = df[numerical_features].corr("spearman")
sns.heatmap(spearman_corr, annot=True, cmap="coolwarm")
plt.title("Spearman Correlation")
plt.savefig(f"{SAVE_DIR}/correlation_spearman.png")
plt.clf()

# --- ANOVA / Kruskal and Boxplots ---
for col in numerical_features:
    groups = [g[col].values for _, g in df.groupby("Class")]
    try:
        stat, p = f_oneway(*groups)
    except:
        stat, p = kruskal(*groups)
    sns.boxplot(data=df, x="Class", y=col)
    plt.title(f"{col} by Class (p = {p:.3e})")
    plt.xticks(rotation=45)
    plt.savefig(f"{SAVE_DIR}/{col}_by_class.png")
    plt.clf()

# --- Amino Acid Frequency ---
aa_counter = Counter("".join(df["Sequence"]))
aa_freq = pd.DataFrame.from_dict(aa_counter, orient="index", columns=["Count"])
aa_freq = aa_freq.sort_values("Count", ascending=False)
sns.barplot(x=aa_freq.index[:20], y=aa_freq["Count"][:20])
plt.title("Top 20 Most Frequent Amino Acids")
plt.savefig(f"{SAVE_DIR}/aa_frequency_top20.png")
plt.clf()

# --- Outlier Check ---
z_scores = df[numerical_features].apply(zscore)
outliers = df[(z_scores.abs() > 3).any(axis=1)]
print(f"Outliers (|z| > 3): {outliers.shape[0]}")

# --- Biological Sanity Check ---
bad_polarity = df[(df["Polar_Proportion"] + df["Nonpolar_Proportion"]) > 1.0]
print(f"Inconsistent polarity rows: {bad_polarity.shape[0]}")
bad_charge = df[(df["Total_Charge"].abs() > 40) & (df["Sequence_Length"] < 100)]
print(f"Extreme charge in short sequences: {bad_charge.shape[0]}")

# --- PCA Projection ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[numerical_features])
y = df['Class']
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, alpha=0.6)
plt.title("PCA (2D) Projection of Numerical Features")
plt.savefig(f"{SAVE_DIR}/pca_projection.png")
plt.clf()

# --- t-SNE on Sample ---
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
df_sample = df.sample(n=2000, random_state=42)
X_sample = scaler.transform(df_sample[numerical_features])
X_tsne = tsne.fit_transform(X_sample)
sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=df_sample["Class"])
plt.title("t-SNE (2D) on Sampled Numerical Features")
plt.savefig(f"{SAVE_DIR}/tsne_projection.png")
plt.clf()

C:\Users\arvyn\AppData\Local\Temp\ipykernel_17164\2291598841.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = [g[col].values for _, g in df.groupby("Class")]
C:\Users\arvyn\AppData\Local\Temp\ipykernel_17164\2291598841.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = [g[col].values for _, g in df.groupby("Class")]
C:\Users\arvyn\AppData\Local\Temp\ipykernel_17164\2291598841.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future defa

Outliers (|z| > 3): 357
Inconsistent polarity rows: 0
Extreme charge in short sequences: 0


<Figure size 640x480 with 0 Axes>

In [10]:
# ------------------ PHASE 2: PREPROCESSING ------------------

# ------------------ Feature Engineering ------------------

# Basic features (original numeric)
base_features = [
    "Molecular_Mass", "Isoelectric_Point", "Hydrophobicity",
    "Total_Charge", "Polar_Proportion", "Nonpolar_Proportion", "Sequence_Length"
]

# Engineered features
df["Charge_Density"] = df["Total_Charge"] / df["Sequence_Length"]
df["Hydrophobicity_x_Polar"] = df["Hydrophobicity"] * df["Polar_Proportion"]
df["Log_Mass"] = np.log1p(df["Molecular_Mass"])
df["PI_per_Length"] = df["Isoelectric_Point"] / df["Sequence_Length"]

# Binned feature: Isoelectric Point group
df["PI_Group"] = pd.cut(df["Isoelectric_Point"], bins=[0, 6.5, 8, 12], labels=["acidic", "neutral", "basic"])

# Flag: extremely short sequence
df["Is_Short_Seq"] = (df["Sequence_Length"] < 100).astype(int)

# Flag: high charge
df["Is_High_Charge"] = (df["Total_Charge"] > 20).astype(int)

# New feature list
engineered_features = [
    "Charge_Density", "Hydrophobicity_x_Polar", "Log_Mass",
    "PI_per_Length", "Is_Short_Seq", "Is_High_Charge"
]

# Include one-hot encoding of PI_Group
df = pd.get_dummies(df, columns=["PI_Group"], drop_first=True)

# Final feature list to use
final_features = base_features + engineered_features + list(df.columns[df.columns.str.startswith("PI_Group_")])

# Preview final engineered features
df_final = df[["Protein_ID", "Class"] + final_features].head()

# ------------------ Label Encoding ------------------
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df["Class"])

# ------------------ Numeric Scaling ------------------
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(df[final_features])

# ------------------ Sequence Tokenization ------------------
tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts(df["Sequence"])
X_seq_tokenized = tokenizer.texts_to_sequences(df["Sequence"])
X_seq_padded = pad_sequences(X_seq_tokenized, maxlen=MAX_LEN, padding='post')

# ------------------ Stratified Train-Validation Split ------------------
X_num_train, X_num_val, X_seq_train, X_seq_val, y_train, y_val = train_test_split(
    X_num_scaled, X_seq_padded, y_encoded,
    test_size=0.2, stratify=y_encoded, random_state=42
)

# ------------------ Save Artifacts ------------------
# Scaler and label encoder
joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.pkl"))
joblib.dump(label_encoder, os.path.join(OUTPUT_DIR, "label_encoder.pkl"))

# Tokenizer
with open(os.path.join(OUTPUT_DIR, "tokenizer.json"), "w") as f:
    f.write(tokenizer.to_json())

# Class mapping
class_mapping = {cls: int(idx) for cls, idx in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))}
with open(os.path.join(OUTPUT_DIR, "class_mapping.json"), "w") as f:
    json.dump(class_mapping, f)

# Engineered features
with open(os.path.join(OUTPUT_DIR, "feature_list.json"), "w") as f:
    json.dump(final_features, f)

# Numpy arrays (optional)
np.save(os.path.join(OUTPUT_DIR, "X_num_train.npy"), X_num_train)
np.save(os.path.join(OUTPUT_DIR, "X_num_val.npy"), X_num_val)
np.save(os.path.join(OUTPUT_DIR, "X_seq_train.npy"), X_seq_train)
np.save(os.path.join(OUTPUT_DIR, "X_seq_val.npy"), X_seq_val)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "y_val.npy"), y_val)

print("✅ Preprocessing complete.")
print(f"Artifacts saved to: {OUTPUT_DIR}")
print(f"Train size: {X_num_train.shape[0]}, Validation size: {X_num_val.shape[0]}")
print(f"Feature count: {X_num_train.shape[1]}")
print(f"Total encoded classes: {len(label_encoder.classes_)}")

✅ Preprocessing complete.
Artifacts saved to: ../Data/preprocessing_artifacts
Train size: 12800, Validation size: 3200
Feature count: 15
Total encoded classes: 5
